In [64]:
import pandas as pd
import numpy as np
import random
import gc
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Image
import requests
from dotenv import load_dotenv
import os
from scipy.sparse import csr_matrix

df_movie = pd.read_csv("data/movies_filtered_augmented.csv")
df_tags = pd.read_csv("data/tags.csv")
df_tmdbID = pd.read_csv("data/links.csv")
df_ratings = pd.read_csv("data/ratings.csv")

#Filtre rating
ratings_movie = df_ratings['movieId'].value_counts()
ratings_movie_top = ratings_movie[ratings_movie > 100].index

condition_FR = df_movie['language'] == "FR"
condition_populaire = df_movie['movieId'].isin(ratings_movie_top)

df_movie = df_movie[condition_FR | condition_populaire].reset_index(drop = True)

film_to_keep = df_movie['movieId']

df_tags = df_tags[df_tags['movieId'].isin(film_to_keep)].reset_index(drop = True)
df_tmdbID = df_tmdbID[df_tmdbID['movieId'].isin(film_to_keep)].reset_index(drop = True)

#Tag
tag_counts = df_tags['tag'].value_counts()
tags_populaires = tag_counts[tag_counts >= 30].index
df_tags_filtre = df_tags[df_tags['tag'].isin(tags_populaires)]

df_genre = df_movie['genres'].str.get_dummies(sep = '|') * 2
if "(no genres listed)" in df_genre.columns:
    df_genre = df_genre.drop(columns="(no genres listed)")
    
df_tags_matrix = pd.crosstab(df_tags_filtre['movieId'], df_tags_filtre['tag'])
df_tags_matrix = np.sqrt(df_tags_matrix) #Sqrt scale (to keep weight of features but not too much)

df_genre['movieId'] = df_movie['movieId']

df_vector = pd.merge(df_genre, df_tags_matrix, on = "movieId", how = 'left').fillna(0)

del_cols = ['movieId']
df_vector = df_vector.drop(columns = del_cols)
df_vector_2D = df_vector.values.astype('float32') #2D Numpy array for better cosine similarity performance

#df
df = df_movie.copy()
df = pd.merge(df_movie, df_tmdbID, on='movieId', how='left')#Add tmdbID for picture and features
df['title'] = df['title'].astype('category')


df_vector_2D = csr_matrix(df_vector_2D)#Sparse matrix to optimize cosine similarity performance

#---------------------------------------------------------------

df_tmdb = pd.read_json("data/movie_ids_08_30_2026.json", lines = True)
df_tmdb = df_tmdb.rename(columns = {'id': 'tmdbId', 'original_title': 'title'})
df_tmdb = df_tmdb[df_tmdb['popularity'] > 3]
df_tmdb = df_tmdb.drop(columns = ['adult', 'video', 'popularity'])

df_merged = pd.merge(df_tmdb, df, on = 'tmdbId', how='left')

df_merged['title_x'] = df_merged['title_x'].astype(str)
df_merged['title_y'] = df_merged['title_y'].astype(str)

df_merged['title'] = df_merged['title_y'].fillna(df_merged['title_x'])
df_merged = df_merged.drop(columns=['title_x', 'title_y'])


print(df_merged.shape)
df_merged.head()

df_merged.to_csv("hello.csv")




(23260, 7)


In [2]:
#Step 2 : ---------- Content engine --------------- (similarity by content)

#User profile

df_favorite_vector = df_vector_2D[[1]]

#Cosine similarity -Manual then Automatic-
'''
#Scalar product
sum_1 = 0
for value in df_favorite_vector.iloc[0]:
    sum_1 += (value)**2

liste_similarity = []

for film in range(len(df_vector)):
    
    scalar_product = 0
    for i in range(len(df_favorite_vector)):
        scalar_product += df_favorite_vector.iloc[i] * df_vector.iloc[film].iloc[i]
    #Magnitude
    sum_2 = 0
    for value in df_vector.iloc[film]:
        sum_2 += (value)**2
    
    magnitude_calculate_df = ((sum_1)**0.5) * ((sum_2)**0.5)
    
    cosine_similarity_manual = scalar_product / magnitude_calculate_df

    liste_similarity.append(cosine_similarity_manual)
'''

#Automatic
similarites = cosine_similarity(df_favorite_vector, df_vector_2D)
df['cosine_similarity'] = similarites[0]

df_sorted = df.sort_values(by=['cosine_similarity'], ascending=False)

print("Film de depart:", df.loc[1, 'title'])
print(df_sorted[['title']].head(5))
    

Film de depart: Jumanji (1995)
                                                  title
1                                        Jumanji (1995)
12349  Chronicles of Narnia: Prince Caspian, The (2008)
1638                                     Flubber (1997)
1965                    Honey, I Shrunk the Kids (1989)
76804                              Jungle Cruise (2021)


In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("API_KEY_TMDB")

In [21]:
tmdbID = df_sorted.iloc[1]['tmdbId']
parameters = {"api_key": API_KEY}

#Provider

URL = f"https://api.themoviedb.org/3/movie/{tmdbID}/watch/providers"
    
reponse = requests.get(URL, params=parameters)
    
if reponse.status_code == 200:
    data = reponse.json()
    
    try:
        provider = data['results']['FR']['flatrate'][0]['provider_name']
    except KeyError:
        provider = "No provider actually available, buy it!"

print(f"See the movie on: {provider}")

#Poster
URL = f"https://api.themoviedb.org/3/movie/{tmdbID}"

reponse = requests.get(URL, params=parameters)
data = reponse.json()

chemin_poster = data['poster_path']

URL_base_image = "https://image.tmdb.org/t/p/w200/"
URL_image_complete = URL_base_image + chemin_poster

reponse_image = requests.get(URL_image_complete)

display(Image(reponse_image.content))

NameError: name 'df_sorted' is not defined

In [74]:
#---------SCRAPPING API---------

import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor


df_merged.reset_index(drop=True, inplace=True)

count = 0

def obtenir_infos_film(tmdb_id):
    global count
    
    URL_movie = f"https://api.themoviedb.org/3/movie/{tmdb_id}?append_to_response=watch/providers"
    URL_tags = f"https://api.themoviedb.org/3/movie/{tmdb_id}/keywords"
    
    language = None
    genre_sentence = None
    provider = None
    vote_count = None
    runtime = None
    release = None
    keyword_sentence = None

    try:
        reponse = requests.get(URL_movie, params=parameters)
        data = reponse.json()
        
        if "origin_country" in data and len(data['origin_country']) > 0:
            language = data['origin_country'][0]

        if "status" in data:
            release = data['status']
        else:
            release = None

        if "runtime" in data:
            runtime = data['runtime']
        else:
            runtime = None

        if "vote_count" in data:
            vote_count = data['vote_count']
        else:
            vote_count = None

        if "genres" in data and len(data["genres"]) > 0:
            genre_list = []
            for genre in data['genres']:
                genre_list.append(genre['name'])
            genre_sentence = ", ".join(genre_list)

        if "watch/providers" in data and "results" in data['watch/providers']:
            results = data['watch/providers']['results']
            if "FR" in results and "flatrate" in results['FR'] and len(results['FR']['flatrate']) > 0:
                provider = results['FR']['flatrate'][0]['provider_name']

    except:
        pass

    try:
        reponse = requests.get(URL_tags, params = parameters)
        data_tags = reponse.json()

        if "keywords" in data_tags and len(data_tags['keywords']) > 0:
            keywords_list = []
            for keyword in data_tags['keywords']:
                keywords_list.append(keyword['name'])
            keyword_sentence = "| ".join(keywords_list)
    except:
        pass

    count += 1
    if count % 250 == 0:
        print(f"{count} / {len(liste_ids)} films traites")

    return {
        'tmdbId': tmdb_id,
        'language': language,
        'provider': provider,
        'genre': genre_sentence,
        'release' : release,
        'runtime' : runtime,
        'vote_count' : vote_count,
        'tags' : keyword_sentence
    }

liste_ids = df_merged['tmdbId'].tolist()

with ThreadPoolExecutor(max_workers=5) as executor:
    resultats = list(executor.map(obtenir_infos_film, liste_ids))

df_infos = pd.DataFrame(resultats)

df_tags = df_infos[['tmdbId', 'tags']]

df_principal = df_infos.drop(columns=['tags'])

df_merged = df_merged.drop(columns=['provider', 'language'], errors='ignore')
df_merged = df_merged.merge(df_principal, on='tmdbId', how='left')

df_merged.to_csv("data/df_merged_scrapped.csv", index=False)
df_tags.to_csv("data/df_tags_tmdb.csv", index=False)

250 / 23302 films traites
500 / 23302 films traites
750 / 23302 films traites
1000 / 23302 films traites
1250 / 23302 films traites
1500 / 23302 films traites
1750 / 23302 films traites
2000 / 23302 films traites
2250 / 23302 films traites
2500 / 23302 films traites
2750 / 23302 films traites
3000 / 23302 films traites
3250 / 23302 films traites
3500 / 23302 films traites
3750 / 23302 films traites
4000 / 23302 films traites
4250 / 23302 films traites
4500 / 23302 films traites
4750 / 23302 films traites
5000 / 23302 films traites
5250 / 23302 films traites
5500 / 23302 films traites
5750 / 23302 films traites
6000 / 23302 films traites
6250 / 23302 films traites
6500 / 23302 films traites
6750 / 23302 films traites
7000 / 23302 films traites
7250 / 23302 films traites
7500 / 23302 films traites
7750 / 23302 films traites
8000 / 23302 films traites
8250 / 23302 films traites
8500 / 23302 films traites
8750 / 23302 films traites
9000 / 23302 films traites
9250 / 23302 films traites
9500

MergeError: Passing 'suffixes' which cause duplicate columns {'provider_y', 'provider_x', 'language_x', 'language_y'} is not allowed.

In [24]:
parameters = {"api_key": "3823bc57ff3c18c2637ab62919ca9bad"}

In [32]:
#Scrap similar movies

import pandas as pd
import requests
import time

df = pd.read_csv("data/df_merged_scrapped.csv")
annees_recentes = ("2026", "2025", "2024", "2023")

count = 0

for index, row in df.iterrows():
    if pd.isna(row["movieId"]):

        movie_id = row["tmdbId"]
        
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/similar"
    
        reponse = requests.get(url, params=parameters)
    
        if reponse.status_code == 200:
      
            data = reponse.json().get("results", [])
    
            for item in data:
                release_date = item.get("release_date", "")
                
                if release_date and not release_date.startswith(annees_recentes):

                    df.loc[index, "similar_id"] = item["id"]
                    df.loc[index, "similar_title"] = item["original_title"]
                    

                    count += 1
                    if count % 250 == 0:
                        print(f"{count} films traites")

                    break

        time.sleep(0.05)

df.to_csv("df_avec_similaires.csv", index=False)

250 films traites
500 films traites
750 films traites
1000 films traites
1250 films traites
1500 films traites
1750 films traites
2000 films traites
2250 films traites
2500 films traites
2750 films traites
3000 films traites
3250 films traites
3500 films traites
3750 films traites
4000 films traites
4250 films traites
4500 films traites
4750 films traites
5000 films traites
5250 films traites
5500 films traites
5750 films traites
6000 films traites
6250 films traites
6500 films traites
6750 films traites
7000 films traites
7250 films traites
7500 films traites
7750 films traites
8000 films traites
8250 films traites
8500 films traites
8750 films traites
9000 films traites
9250 films traites


In [36]:
df = pd.read_csv("data/df_merged_scrapped.csv")
df_ratings = pd.read_csv("data/ratings_augmented.csv")

df_new = pd.merge(df, df_ratings[["movie_idx", "movieId"]].drop_duplicates(), on="movieId", how="left")

df_new.to_csv("hello.csv")